<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 11 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Publish a Semantic View and Its Features</div>
</div>

A dashboard reads daily orders and a model reads features; both need an interface whose rows and meaning stay the same. You publish `bi_order_metrics_l2`, a semantic view with one row per day over Lab 10's service table, project features over every day of the reporting window so that a day without orders appears, check rows, nulls, dates, and totals before publication, and make a broken contract and a stale snapshot fail on purpose. The optional sections 5 and 6 start a local Superset and build a three-chart dashboard on the case's payment contract, read by a Doris account that can read only that view. Plan about 70 minutes; the optional sections add about 15, and the first download.

**Module 11: Interfaces and Boundaries for BI and AI Consumers** · [Lesson](course.md) · [Quiz](quiz11_bi_and_ai.ipynb) · [Course README](../../README.md)

## Before you begin

Complete Lab 10 first, in the same database. This Lab reads its service table `daily_order_metrics_l2`, and the optional sections its view `bi_paid_kpis_l2`; it changes neither.

The Lab rebuilds the view `bi_order_metrics_l2`, which Module 12 publishes, and in the practice `bi_order_metrics_by_region_l2`; step 4 creates `bi_contract_broken_l2` and drops it again.

The optional sections 5 and 6 run only when you switch them on, and they create things outside your lab database that stay after the Lab, so that the dashboard keeps working: the Compose project `doris-warehousing-bi` with the container `doris-warehousing-bi-superset-1` and the volume of its metadata, the image `doris-course-superset:4.1.2-pydoris1.1.0`, built on your computer, and the Doris account `course_bi_<suffix>`, which can read only `bi_paid_kpis_l2`. Module 11's lesson, section 11.6, says how to remove them. Skipping the optional sections changes nothing for the quiz or for Level 3.

This Lab installs no MCP Server and calls no model; the lesson explains how an AI tool's access to Doris is bounded.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, expected_failure
from dw_course.ui import card, show_sql

lab = connect_sandbox(module=11)

## 1. Publish a semantic view

A semantic view is an interface for consumers: stable names, one declared grain, and measures whose meaning is written down. `bi_order_metrics_l2` rolls Lab 10's rows up to one row per day: `order_date` is the dimension; `order_count` and `gross_amount` are measures that can be added up over days; `average_order_amount` is a ratio, computed from the day's sums, and `NULL` when a day has no orders. A dashboard must not add up or average the ratio over days: it adds up the sums and divides again.

In [ ]:
lab.execute("DROP VIEW IF EXISTS bi_order_metrics_l2")
view_ddl = """CREATE VIEW bi_order_metrics_l2 AS
SELECT order_date,
       SUM(order_count) AS order_count,
       SUM(gross_amount) AS gross_amount,
       CASE WHEN SUM(order_count) = 0 THEN NULL
            ELSE SUM(gross_amount) / SUM(order_count) END AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date"""
show_sql("The semantic view", view_ddl)
lab.execute(view_ddl)
lab.sql("SELECT * FROM bi_order_metrics_l2 ORDER BY order_date", title="The semantic view: one row per day")
expect(lab.query("SELECT CAST(order_date AS STRING), order_count, gross_amount, average_order_amount FROM bi_order_metrics_l2 ORDER BY order_date"),
       [("2026-01-01", 9, "1200.00", "133.33"), ("2026-01-02", 1, "110.00", "110.00")],
       title="Two days, as the contract says")

**Expected result**

The definition, then two rows: 2026-01-01 with 9 orders, 1200.00, and an average of 133.333333, and 2026-01-02 with 1 order, 110.00, and 110.000000. A green check, which compares the averages to two decimals.

The view stores nothing: each query reads the service table. Its column names and their meaning are the contract; Lab 10's table can change, for example get another column, without changing what the view's consumers read.

## 2. Project features over the reporting window

A model that reads daily features needs a row for every day, also a day without orders: a missing row and a day with zero orders are not the same thing, and only the second is a feature. The semantic view has rows only for the days that had orders. The projection therefore starts from a calendar of the reporting window, 2026-01-01 to 2026-01-03, made with the table function `numbers()`, and joins the view to it.

`zero_order_flag` is 1 for a day without orders. The average stays `NULL` on such a day, because a ratio with no orders has no value; the counts become 0. Predict how many rows the projection has, and which day gets the flag.

In [ ]:
features_sql = """WITH calendar AS (
    SELECT DATE_ADD(DATE '2026-01-01', number) AS feature_date FROM numbers("number"="3")
)
SELECT CAST(c.feature_date AS STRING) AS feature_date,
       COALESCE(m.order_count, 0) AS feature_order_count,
       COALESCE(m.gross_amount, 0.00) AS feature_gross_amount,
       m.average_order_amount AS feature_average_amount,
       CASE WHEN COALESCE(m.order_count, 0) = 0 THEN 1 ELSE 0 END AS zero_order_flag
FROM calendar c
LEFT JOIN bi_order_metrics_l2 m ON m.order_date = c.feature_date
ORDER BY c.feature_date"""
lab.sql(features_sql, title="Features for every day of the reporting window")
expect([(row[0], row[1], row[4]) for row in lab.query(features_sql)],
       [("2026-01-01", 9, 0), ("2026-01-02", 1, 0), ("2026-01-03", 0, 1)],
       title="Three days; the day without orders has the flag")

**Expected result**

Three rows and a green check: 2026-01-01 with 9 orders, 1200.00, an average of 133.333333, and flag 0; 2026-01-02 with 1 order, 110.00, 110.000000, and flag 0; 2026-01-03 with 0 orders, 0.00, no average (`NULL`), and flag 1.

The projection prepares inputs; it says nothing about a model's accuracy. Whether January 3 had no orders or its orders have not arrived yet, the data cannot tell: the freshness check of step 4 and the pipeline's own records have to.

## 3. Check the interface before publication

Before a dashboard or a model reads the view, check what its consumers rely on: the number of days, measures that are never `NULL`, the first and the last date, and the total orders and amount against the service table. These checks hold the interface to its contract; a chart's filters and a model's behavior are checked by their own owners.

In [ ]:
checks_sql = """SELECT COUNT(*) AS serving_days,
       SUM(CASE WHEN order_count IS NULL OR gross_amount IS NULL THEN 1 ELSE 0 END) AS days_with_nulls,
       CAST(MIN(order_date) AS STRING) AS first_date,
       CAST(MAX(order_date) AS STRING) AS last_date,
       SUM(order_count) AS served_orders,
       SUM(gross_amount) AS served_amount
FROM bi_order_metrics_l2"""
lab.sql(checks_sql, title="The interface's checks")
expect(lab.query(checks_sql), [(2, 0, "2026-01-01", "2026-01-02", 10, "1310.00")], title="The interface passes its checks")
expect(lab.query("SELECT SUM(order_count), SUM(gross_amount) FROM daily_order_metrics_l2"), [(10, "1310.00")],
       title="The view adds up to the service table")

**Expected result**

One row: 2 serving days, 0 days with nulls, first date 2026-01-01, last date 2026-01-02, 10 orders, and 1310.00; two green checks, the second against Lab 10's service table.

## 4. Make a broken contract and a stale snapshot fail

A query that runs is not a contract test. The cell first reads the view's column names through a cursor and compares them with the four that the contract publishes. Then it creates a broken copy that leaves out `average_order_amount`: the same check must reject it. Finally it measures freshness against fixed teaching dates, since the course's orders are from January 2026: the latest day is January 2, one day old on January 3, which a one-day rule allows, and seven days old on January 9, which it does not. A real check uses the business's clock and its declared rule.

In [ ]:
required_columns = ["order_date", "order_count", "gross_amount", "average_order_amount"]

def published_columns(view_name):
    with lab.connection.cursor() as cursor:
        cursor.execute(f"SELECT * FROM {view_name} LIMIT 0")
        return [column[0] for column in cursor.description]

expect(published_columns("bi_order_metrics_l2"), required_columns, title="The view publishes the four columns of the contract")
lab.execute("DROP VIEW IF EXISTS bi_contract_broken_l2")
try:
    lab.execute("CREATE VIEW bi_contract_broken_l2 AS SELECT order_date, order_count, gross_amount FROM bi_order_metrics_l2")
    with expected_failure("The contract check catches a missing measure", "A consumer that reads average_order_amount would fail on this view"):
        expect(published_columns("bi_contract_broken_l2"), required_columns)
finally:
    lab.execute("DROP VIEW IF EXISTS bi_contract_broken_l2")

freshness_sql = """
SELECT CAST(MAX(order_date) AS STRING) AS latest_date,
       DATEDIFF(%s, MAX(order_date)) AS lag_days
FROM bi_order_metrics_l2
"""
lab.sql(freshness_sql, ("2026-01-03",), title="The lag on January 3")
expect(lab.query(freshness_sql, ("2026-01-03",)), [("2026-01-02", 1)], title="One day old: within the one-day rule")
with expected_failure("The freshness check catches a stale snapshot", "Seven days old on January 9: more than the one-day rule allows"):
    expect(lab.query(freshness_sql, ("2026-01-09",))[0][1] <= 1, True)

**Expected result**

Four cards. A green check: the view publishes `order_date`, `order_count`, `gross_amount`, and `average_order_amount`. A green card: the contract check caught the missing measure of the broken copy, which is then dropped. The lag on January 3: latest date 2026-01-02, 1 day, and a green check. A green card: seven days old on January 9 is more than the rule allows.

The two expected failures are the point of the step: a check that has never failed may check nothing. Neither changed `bi_order_metrics_l2`.

## 5. Optional: connect a real dashboard to the payment contract

**This section and section 6 are optional, and they run only when you switch them on**: set `BUILD_DASHBOARD = True` in the next cell, then run it. They need internet access for the first image build, about 2 GiB of additional Docker memory, and the host port 51888 (`DW_BI_PORT`). Skipping them does not affect Quiz 11 or Level 3; continue with the practice. Module 11's lesson, section 11.6, lists what they create and how to remove it.

Complete Lab 10 sections 5 and 6 first; they build `bi_paid_kpis_l2`. A Notebook table is not yet a Business Intelligence (BI) dashboard. This section starts **Apache Superset 4.1.2**, registers the Doris SQLAlchemy driver, and creates three real charts over `bi_paid_kpis_l2`:

1. Payment-window GMV, with refunds shown separately in the SQL contract.
2. Distinct paid orders in the payment window.
3. Created-order cohort conversion, using its reviewed numerator and denominator.

The helper provisions configuration through Superset's API; it also executes each chart's query through Superset and checks the returned value. Open the generated dashboard URL to inspect the actual rendering, metric SQL and dataset. Do not treat API value checks as proof of browser layout, freshness scheduling, or a production SLA.

**Service scope and first-start cost:** the course-owned Superset container is capped at 2 GiB RAM and two CPUs, binds only `127.0.0.1:51888`, and retains its SQLite metadata volume. To use another port, start Jupyter with `DW_BI_PORT` set, for example `DW_BI_PORT=61888 .venv/bin/jupyter lab`; the helper checks the port before it starts anything and names the program to stop when it is busy. The first image build needs internet access and may take several minutes. This is a local teaching setup with public UI credentials (`course_admin` / `course_local_only`), a fixed local secret and no production TLS; do not expose it to a public network. The database connection uses a different generated **view-only Doris account**, not `root` and not the UI administrator.

If Jupyter runs on a remote Linux machine, forward the BI port as well as the Jupyter port: `ssh -N -L 51888:127.0.0.1:51888 user@host`. If 51888 is already occupied, stop the old tunnel or choose another local port; do not terminate an unrelated service.

If the first start fails, the message says why: the image could not be downloaded or built (the machine running Docker needs internet access), Docker ran out of memory, or the service did not become healthy within 5 minutes; its log is in the collapsed details.

In [ ]:
# Optional: set to True, then run this cell, to start Superset and build the dashboard.
BUILD_DASHBOARD = False

if BUILD_DASHBOARD:
    from dw_course.bi import build_payment_dashboard
    payment_dashboard = build_payment_dashboard(lab, start=True)
else:
    card("Set BUILD_DASHBOARD = True in this cell and run it again to start Superset and build the dashboard.",
         "skip", "The optional dashboard was not built")

**Expected result**

With `BUILD_DASHBOARD = False`: a gray card, the optional dashboard was not built.

With `True`: a green check that the payment contract holds (660.00 paid GMV, 5 paid orders, 5 of 8 created orders converted); the three steps of the service's start, all green (the first start builds the image and takes a few minutes); a green check that the dashboard's Doris account can read `bi_paid_kpis_l2` (the helper also checks that it cannot read `dwd_payments_l2`, and stops if it can); three green checks of the values that Superset itself returned, 660.0, 5.0, and 0.625; a green check that the three charts belong to the dashboard; then the dashboard's URL, `http://127.0.0.1:51888/superset/dashboard/<id>/` (or the port in `DW_BI_PORT`), the UI user `course_admin`, the Doris reader `course_bi_<suffix>`, and the cutoff 2026-01-04 00:00:00.

## 6. Optional: review what the consumer can do

Open the URL above, sign in with the local UI account, and inspect all three charts. The conversion chart should display a percentage, not an order count. In Superset's dataset and chart editors, trace each metric back to its visible SQL contract. The cutoff is a fixed teaching timestamp; refreshing the dashboard does not make that timestamp current.

The generated Doris reader can read only `bi_paid_kpis_l2`; the helper verifies that a direct read of `dwd_payments_l2` is denied. Turning off SQL Lab and DML in Superset is useful, but database grants are the actual boundary. The local Superset UI administrator can change configuration; this is not a demonstration of multi-tenant UI permissions.

The service, charts and view-only account remain available for review. Rerunning updates only this database's course-owned BI objects and rotates the reader password. Stop the service from the course root with `docker compose --project-name doris-warehousing-bi --file environments/bi/compose.yml stop`; metadata is retained. Drop the generated Doris account separately only when retiring this dashboard. Do not use `down -v` as ordinary cleanup.

## 7. Practice: add a region filter without breaking consumers

The BI team asks to filter the daily orders by region. `bi_order_metrics_l2` has no region, and adding one would change its grain: a row would no longer be a day. Publish a second view, `bi_order_metrics_by_region_l2`, with one row per day and region, from the same service table, which has that grain already. Check that its rows add up to the first view's days, and that the first view still publishes the same four columns.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.execute("DROP VIEW IF EXISTS bi_order_metrics_by_region_l2")
lab.execute("""CREATE VIEW bi_order_metrics_by_region_l2 AS
SELECT order_date, region,
       SUM(order_count) AS order_count,
       SUM(gross_amount) AS gross_amount,
       CASE WHEN SUM(order_count) = 0 THEN NULL
            ELSE SUM(gross_amount) / SUM(order_count) END AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date, region""")
lab.sql("SELECT * FROM bi_order_metrics_by_region_l2 ORDER BY order_date, region", title="One row per day and region")
by_day = """SELECT CAST(order_date AS STRING), SUM(order_count), SUM(gross_amount)
FROM bi_order_metrics_by_region_l2 GROUP BY order_date ORDER BY order_date"""
expect(lab.query(by_day), lab.query("SELECT CAST(order_date AS STRING), order_count, gross_amount FROM bi_order_metrics_l2 ORDER BY order_date"),
       title="The regions add up to the days of bi_order_metrics_l2")
expect(published_columns("bi_order_metrics_l2"), required_columns, title="bi_order_metrics_l2 keeps its four columns")

**Expected result**

Three rows: 2026-01-01 EAST with 5 orders, 720.00, and 144.000000; 2026-01-01 WEST with 4, 480.00, and 120.000000; 2026-01-02 EAST with 1, 110.00, and 110.000000. Two green checks: the regions add up to the first view's days, and `bi_order_metrics_l2` keeps its four columns.

The new view has its own grain, so it is a second contract: it needs its own refresh, access, and checks, and the dashboards of the first view keep working unchanged. Notice that the daily average of 133.33 is not the average of 144.00 and 120.00: each view divides its own sums.

## You are done when

- `bi_order_metrics_l2` publishes two days, 10 orders, and 1310.00, and you can say which of its columns are dimensions, measures, and ratios;
- you can explain why the feature projection starts from a calendar, and what `zero_order_flag` on January 3 does and does not say;
- you have seen the contract check and the freshness check fail on purpose, and pass on the published view;
- you can say what a second view, with its own grain, changes for consumers and what it does not.

## Next

Open [Quiz 11: Interfaces and Boundaries for BI and AI Consumers](quiz11_bi_and_ai.ipynb). This is the last Lab of Level 2; Level 3 starts with [Module 12: Publishing, Permissions, and Audit](../../level3/module12-publishing-permissions-audit/course.md).